In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# Figures and statistics for the manuscript. Run AFTER nb6 has finished.
# Reads only what nb6 wrote to Drive, so it needs no GPU and no model code.
from qct3d.config import mount_drive_if_colab
mount_drive_if_colab(P)


In [ ]:
# ==============================================================================================
# FIG-1. Load every result the overnight runs wrote, and say what is present
# ==============================================================================================
# Reads overnight_results.jsonl (the append-only log, last record per key wins) plus, if they
# exist, the Part C transfer outputs from nb5. Produces publication figures at 300 dpi as both
# PNG and PDF, and prints the exact numbers under each one so they can be quoted in the text
# without being re-derived by hand.
import json
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

OUTPUT_DIR = P["rsna_output_dir"]
INDIA_DIR = P["external_output_dir"]
FIG_DIR = os.path.join(OUTPUT_DIR, "figures_paper")
os.makedirs(FIG_DIR, exist_ok=True)

# ---- house style -------------------------------------------------------------------------
# Three series only, in fixed slot order, from a palette validated for colour-vision
# deficiency at all pairs (worst CVD dE 9.2, worst normal-vision dE 24.0). The aqua sits below
# 3:1 against a light surface, so every series is DIRECTLY LABELLED as well as legended --
# identity is never carried by colour alone.
C_QUANTUM, C_CLASSICAL, C_LINEAR = "#2a78d6", "#eb6834", "#1baf7a"
ARM_COLOR = {"quantum": C_QUANTUM, "classical": C_CLASSICAL, "linear_probe": C_LINEAR}
ARM_LABEL = {"quantum": "quantum", "classical": "classical (matched)",
             "linear_probe": "linear probe"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"

mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 8.5, "axes.titlesize": 9.5, "axes.labelsize": 8.5,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK,
    "axes.titlecolor": INK, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "legend.fontsize": 7.5, "lines.linewidth": 2.0,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"))
    print(f"  -> {os.path.join(FIG_DIR, name)}.png / .pdf")


def load_log(path=None):
    path = path or os.path.join(OUTPUT_DIR, "overnight_results.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run nb6 first.")
    latest = {}
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except Exception:
                continue
            if r.get("key"):
                latest[r["key"]] = r      # append-only, last wins
    return pd.DataFrame(list(latest.values()))


LOG = load_log()
OK = (LOG[LOG.get("auroc").notna()] if "auroc" in LOG.columns else LOG.iloc[:0]).copy()
print(f"{len(LOG):,} unit(s) on the log, {len(OK):,} with a score")
if "error" in LOG.columns and LOG["error"].notna().any():
    print(f"  !! {int(LOG['error'].notna().sum())} unit(s) recorded an error")
for st in sorted(OK["stage"].dropna().unique()):
    _s = OK[OK["stage"] == st]
    print(f"  {st:6s} {len(_s):>6,} unit(s)   backbones {sorted(_s['backbone'].dropna().unique())}")
# JSON gives Python bools, which pandas keeps as OBJECT dtype -- and `~` on an object column
# is bitwise-not, so ~True becomes -2 and the mask silently turns into column indexing. Coerce
# the types once, here, rather than defending against it at every use site.
for _c, _t in (("encoder_norm", bool), ("seed", "Int64"), ("k", "Int64"), ("draw", "Int64")):
    if _c in OK.columns:
        OK[_c] = (OK[_c].astype(bool) if _t is bool
                  else pd.to_numeric(OK[_c], errors="coerce").astype("Int64"))

# STALE ROWS FROM THE FIRST NIGHT'S RUN.
# The first E0 keys were E0|backbone|family|reducer|norm|arm -- no placement, no seed -- and
# they were produced before `fit_arm` forwarded `family`, so all of them are secretly the same
# ansatz wearing three different labels. The second night's keys differ, so BOTH sets survive
# in the append-only log. They happen to be excluded by every figure below because each one
# filters on `placement`, which those rows do not have -- but that is luck, not a design, and
# the next figure someone writes will not know. Drop them here, loudly.
_stale = OK[(OK["stage"] == "E0") & (OK.get("placement").isna()
                                     if "placement" in OK.columns else True)]
if len(_stale):
    print(f"\n  dropping {len(_stale)} E0 row(s) from the superseded first run "
          f"(no placement/seed recorded; family label was not honoured in that build)")
    OK = OK.drop(index=_stale.index)

E0 = OK[OK["stage"] == "E0"].copy()
E6 = OK[OK["stage"] == "E6"].copy()
E7 = OK[OK["stage"] == "E7"].copy()
SHORT = {"B1_swinunetr_suprem": "B1 SwinUNETR", "B2_segresnet_suprem": "B2 SegResNet",
         "B3_resnet18_medicalnet": "B3 ResNet-18"}
E0["bb"] = E0["backbone"].map(SHORT).fillna(E0["backbone"])
for _d in (E6, E7):
    _d["bb"] = _d["backbone"].map(SHORT).fillna(_d["backbone"])
print(f"\nE0 cells: {E0.groupby(['backbone','placement','family','encoder_norm','arm']).ngroups}"
      f"  seeds per cell: {sorted(E0['seed'].dropna().unique())}")


In [ ]:
# ==============================================================================================
# FIG-2. Figure 1 -- the saturation cliff, and what conditioning recovers
# ==============================================================================================
# Left: every pooled cell as a point, median pre-tanh magnitude (log) against AUROC. The
# x-axis is the diagnostic the paper contributes, so it is the axis. Right: the same cells as
# paired slopes, unconditioned to conditioned, which is how a reader sees that the recovery is
# universal rather than an average over a few big movers.
_p = E0[(E0["placement"] == "pooled")].copy()
_g = (_p.groupby(["bb", "family", "encoder_norm", "arm"])
      .agg(auroc=("auroc", "mean"), sd=("auroc", "std"),
           pretanh=("pretanh_after", "median")).reset_index())

fig, (axL, axR) = plt.subplots(1, 2, figsize=(7.4, 3.1))
fig.subplots_adjust(wspace=0.34, right=0.84)

for _arm, _c in (("quantum", C_QUANTUM), ("classical", C_CLASSICAL)):
    _s = _g[_g["arm"] == _arm]
    axL.scatter(_s["pretanh"], _s["auroc"], s=26, color=_c, zorder=3,
                edgecolor="white", linewidth=0.8, label=ARM_LABEL[_arm])
axL.axvspan(5, 2000, color="#f2f1ee", zorder=0)
axL.axvline(5, color=INK2, lw=0.8, ls=(0, (3, 3)), zorder=1)
# axes-fraction coordinates: the data range changes between backbones and targets, and a
# hard-coded y puts this annotation outside the panel the moment it does.
axL.text(0.97, 0.04, "shaded: tanh flat\n(saturated)", transform=axL.transAxes,
         fontsize=7, color=INK2, ha="right", va="bottom")
axL.set_xscale("log")
axL.set_xlabel("median |pre-tanh| after training")
axL.set_ylabel("AUROC (RSNA holdout)")
axL.set_title("a  saturation tracks AUROC", loc="left")
axL.grid(axis="y", zorder=0)
axL.set_axisbelow(True)
axL.legend(loc="upper right", handletextpad=0.4)

_w = _g.pivot_table(index=["bb", "family", "arm"], columns="encoder_norm",
                    values="auroc").reset_index()
for _, _r in _w.iterrows():
    _c = ARM_COLOR[_r["arm"]]
    axR.plot([0, 1], [_r[False], _r[True]], color=_c, lw=1.4, alpha=0.75, zorder=2)
    axR.scatter([0, 1], [_r[False], _r[True]], s=18, color=_c, zorder=3,
                edgecolor="white", linewidth=0.7)
axR.set_xticks([0, 1])
axR.set_xticklabels(["no norm", "+ BatchNorm\n(0 parameters)"])
axR.set_xlim(-0.22, 1.75)
axR.set_ylabel("AUROC")
axR.set_title("b  every cell recovers", loc="left")
axR.grid(axis="y")
axR.set_axisbelow(True)
# direct labels rather than a second legend -- identity is never colour alone
for _arm, _dy in (("quantum", 0.004), ("classical", -0.010)):
    _v = _w[_w["arm"] == _arm][True].mean()
    axR.text(1.06, _v + _dy, ARM_LABEL[_arm], color=ARM_COLOR[_arm], fontsize=7.5, va="center")
save(fig, "F1_saturation")
plt.show()

_lift = (_w[True] - _w[False])
print("\nFIGURE 1 NUMBERS")
print(f"  pre-tanh, unconditioned : median {_g[_g['encoder_norm'] == False]['pretanh'].median():.0f}"
      f"  range {_g[_g['encoder_norm'] == False]['pretanh'].min():.0f}-"
      f"{_g[_g['encoder_norm'] == False]['pretanh'].max():.0f}")
print(f"  pre-tanh, conditioned   : median {_g[_g['encoder_norm'] == True]['pretanh'].median():.2f}"
      f"  range {_g[_g['encoder_norm'] == True]['pretanh'].min():.2f}-"
      f"{_g[_g['encoder_norm'] == True]['pretanh'].max():.2f}")
_lq = _lift[_w["arm"] == "quantum"]
_lc = _lift[_w["arm"] == "classical"]
print(f"  AUROC lift, QUANTUM arms   : mean +{_lq.mean():.4f}  "
      f"range +{_lq.min():.4f} to +{_lq.max():.4f}  (n={len(_lq)} cells)")
print(f"  AUROC lift, CLASSICAL arms : mean +{_lc.mean():.4f}  "
      f"range +{_lc.min():.4f} to +{_lc.max():.4f}  (n={len(_lc)} cells)")
print(f"  AUROC lift, all cells      : mean +{_lift.mean():.4f}"
      f"   ({int(E0['seed'].nunique())} seeds per cell)")
print("  QUOTE ONE OF THESE CONSISTENTLY. The all-cells mean is lower than the quantum mean")
print("  only because the classical arms saturate slightly less, not because the effect is")
print("  weaker -- say which arm a number refers to every time it appears.")


In [ ]:
# ==============================================================================================
# FIG-3. Figure 2 -- the ansatz ranking is a saturation ranking
# ==============================================================================================
# The claim is a comparison between an effect and the noise it must exceed, so the figure plots
# both on one axis: the spread across ansatze within a backbone, against the seed-to-seed
# standard deviation of the same cells. Above the diagonal the ansatz looks real; below it, it
# is indistinguishable from re-running with a different seed.
_q = E0[(E0["placement"] == "pooled") & (E0["arm"] == "quantum")]
_rows = []
for (_bb, _nm), _s in _q.groupby(["bb", "encoder_norm"]):
    _m = _s.groupby("family")["auroc"].mean()
    _sd = _s.groupby("family")["auroc"].std()
    _rows.append(dict(bb=_bb, norm=bool(_nm), spread=_m.max() - _m.min(),
                      seed_sd=_sd.mean(), lo=_m.min(), hi=_m.max()))
_R = pd.DataFrame(_rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 3.1))
fig.subplots_adjust(wspace=0.32)
_lim = max(_R["spread"].max(), _R["seed_sd"].max()) * 1.18
ax1.plot([0, _lim], [0, _lim], color=INK2, lw=0.9, ls=(0, (3, 3)), zorder=1)
ax1.text(_lim * 0.97, _lim * 0.9, "effect = noise", fontsize=7, color=INK2, ha="right")
for _nm, _c, _lab in ((False, C_CLASSICAL, "unconditioned"), (True, C_QUANTUM, "conditioned")):
    _s = _R[_R["norm"] == _nm]
    ax1.scatter(_s["seed_sd"], _s["spread"], s=42, color=_c, zorder=3,
                edgecolor="white", linewidth=0.9, label=_lab)
    for _, _r in _s.iterrows():
        ax1.annotate(_r["bb"].split()[0], (_r["seed_sd"], _r["spread"]),
                     textcoords="offset points", xytext=(6, -1), fontsize=7, color=_c)
ax1.set_xlim(0, _lim); ax1.set_ylim(0, _lim)
ax1.set_xlabel("seed-to-seed SD of the same cells")
ax1.set_ylabel("spread across Q1 / Q2 / Q4")
ax1.set_title("a  effect vs training noise", loc="left")
ax1.grid(); ax1.set_axisbelow(True); ax1.legend(loc="upper left", handletextpad=0.4)

_fams = ["Q1_sel", "Q2_reupload", "Q4_ttn"]
_x = np.arange(len(_fams))
for _i, (_nm, _c, _lab) in enumerate(((False, C_CLASSICAL, "unconditioned"),
                                      (True, C_QUANTUM, "conditioned"))):
    for _j, _bb in enumerate(sorted(_q["bb"].unique())):
        _s = _q[(_q["encoder_norm"] == _nm) & (_q["bb"] == _bb)]
        _m = [_s[_s["family"] == _f]["auroc"].mean() for _f in _fams]
        _e = [_s[_s["family"] == _f]["auroc"].std() for _f in _fams]
        ax2.errorbar(_x + (_j - 1) * 0.11, _m, yerr=_e, color=_c, lw=1.4, marker="o",
                     ms=4, capsize=2, alpha=0.9, label=_lab if _j == 1 else None)
ax2.set_xticks(_x); ax2.set_xticklabels(["Q1\nentangling", "Q2\nre-upload", "Q4\ntree TN"])
ax2.set_ylabel("AUROC")
ax2.set_title("b  the ranking disappears", loc="left")
ax2.grid(axis="y"); ax2.set_axisbelow(True)
ax2.text(2.28, _q[_q["encoder_norm"] == True]["auroc"].mean(), "conditioned",
         color=C_QUANTUM, fontsize=7.5, va="center")
ax2.text(2.28, _q[_q["encoder_norm"] == False]["auroc"].mean(), "unconditioned",
         color=C_CLASSICAL, fontsize=7.5, va="center")
ax2.set_xlim(-0.35, 3.25)
save(fig, "F2_ansatz_is_saturation")
plt.show()

print("\nFIGURE 2 NUMBERS")
for _, _r in _R.sort_values(["bb", "norm"]).iterrows():
    print(f"  {_r['bb']:14s} {'conditioned  ' if _r['norm'] else 'unconditioned'}"
          f"  spread {_r['spread']:.4f}  seed sd {_r['seed_sd']:.4f}"
          f"  ratio {_r['spread'] / max(_r['seed_sd'], 1e-9):.1f}x")


In [ ]:
# ==============================================================================================
# FIG-4. Figure 3 -- placement, and quantum vs its parameter-matched control: two nulls
# ==============================================================================================
# Both panels are difference plots with an interval, because both claims are nulls and a null
# is only readable as an interval that contains zero. Intervals are t-based over the seeds,
# which is the variation the claim is about: the two arms in each pair see identical splits,
# so test-set sampling is common to both and cancels.
from scipy import stats as _st

_cond = E0[E0["encoder_norm"] == True]                                        # noqa: E712


def _seed_delta(sub_a, sub_b):
    """Per-seed difference, then a t interval. Same splits on both sides."""
    _a = sub_a.groupby("seed")["auroc"].mean()
    _b = sub_b.groupby("seed")["auroc"].mean()
    _d = (_a - _b).dropna()
    if len(_d) < 2:
        return float(_d.mean()) if len(_d) else np.nan, np.nan, np.nan, len(_d)
    _se = _d.std(ddof=1) / np.sqrt(len(_d))
    _t = _st.t.ppf(0.975, len(_d) - 1)
    return float(_d.mean()), float(_d.mean() - _t * _se), float(_d.mean() + _t * _se), len(_d)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 3.2))
fig.subplots_adjust(wspace=0.42)

_rows = []
for _bb in sorted(_cond["bb"].unique()):
    _sp = _cond[(_cond["bb"] == _bb) & (_cond["placement"] == "spatial")
                & (_cond["arm"] == "quantum")]
    _po = _cond[(_cond["bb"] == _bb) & (_cond["placement"] == "pooled")
                & (_cond["family"] == "Q1_sel") & (_cond["arm"] == "quantum")]
    if len(_sp) and len(_po):
        _m, _lo, _hi, _n = _seed_delta(_sp, _po)
        _rows.append(dict(label=_bb, m=_m, lo=_lo, hi=_hi, n=_n))
_P = pd.DataFrame(_rows)
_y = np.arange(len(_P))
ax1.axvline(0, color=INK2, lw=0.9, zorder=1)
ax1.errorbar(_P["m"], _y, xerr=[_P["m"] - _P["lo"], _P["hi"] - _P["m"]], fmt="o",
             color=C_QUANTUM, ms=5, lw=1.6, capsize=3, zorder=3)
ax1.set_yticks(_y); ax1.set_yticklabels(_P["label"])
ax1.set_xlabel("$\\Delta$AUROC,  spatial $-$ pooled")
ax1.set_title("a  placement (both conditioned)", loc="left")
ax1.grid(axis="x"); ax1.set_axisbelow(True); ax1.set_ylim(-0.6, len(_P) - 0.4)

_rows = []
for (_bb, _pl, _fa), _s in _cond.groupby(["bb", "placement", "family"]):
    _q = _s[_s["arm"] == "quantum"]; _c = _s[_s["arm"] == "classical"]
    if len(_q) and len(_c):
        _m, _lo, _hi, _n = _seed_delta(_q, _c)
        _rows.append(dict(label=f"{_bb.split()[0]} {'spatial' if _pl == 'spatial' else _fa.split('_')[0]}",
                          m=_m, lo=_lo, hi=_hi))
_Q = pd.DataFrame(_rows).sort_values("label").reset_index(drop=True)
_y = np.arange(len(_Q))
ax2.axvline(0, color=INK2, lw=0.9, zorder=1)
ax2.errorbar(_Q["m"], _y, xerr=[_Q["m"] - _Q["lo"], _Q["hi"] - _Q["m"]], fmt="o",
             color=C_QUANTUM, ms=4.5, lw=1.4, capsize=2.5, zorder=3)
ax2.set_yticks(_y); ax2.set_yticklabels(_Q["label"], fontsize=7)
ax2.set_xlabel("$\\Delta$AUROC,  quantum $-$ classical")
ax2.set_title("b  quantum vs its control", loc="left")
ax2.grid(axis="x"); ax2.set_axisbelow(True); ax2.set_ylim(-0.6, len(_Q) - 0.4)
save(fig, "F3_two_nulls")
plt.show()

print("\nFIGURE 3 NUMBERS")
print("  placement (spatial - pooled), 95% t interval over seeds:")
for _, _r in _P.iterrows():
    print(f"    {_r['label']:14s} {_r['m']:+.4f}  [{_r['lo']:+.4f}, {_r['hi']:+.4f}]")
print(f"    mean across backbones {_P['m'].mean():+.4f}")
print("  quantum - classical:")
for _, _r in _Q.iterrows():
    print(f"    {_r['label']:16s} {_r['m']:+.4f}  [{_r['lo']:+.4f}, {_r['hi']:+.4f}]")
print(f"    mean {_Q['m'].mean():+.4f}   max |delta| {_Q['m'].abs().max():.4f}"
      f"   intervals containing 0: {int(((_Q['lo'] <= 0) & (_Q['hi'] >= 0)).sum())}/{len(_Q)}")


In [ ]:
# ==============================================================================================
# FIG-5. Figure 4 -- label efficiency on the Indian cohort, and Figure 5 -- transfer
# ==============================================================================================
# The published claim these test is about the SHAPE of a curve as labels get scarce, so the
# curve is the figure and every point carries the spread over its draws. One panel per
# backbone; three arms, each directly labelled at the right-hand end as well as legended.
def _curve_panel(ax, d, title, show_legend=False, label_right=False):
    for _arm in ("classical", "quantum", "linear_probe"):
        _s = d[d["arm"] == _arm]
        if not len(_s):
            continue
        _g = _s.groupby("k")["auroc"].agg(["mean", "std", "size"]).reset_index()
        _c = ARM_COLOR[_arm]
        ax.fill_between(_g["k"], _g["mean"] - _g["std"], _g["mean"] + _g["std"],
                        color=_c, alpha=0.13, lw=0, zorder=2)
        ax.plot(_g["k"], _g["mean"], color=_c, marker="o", ms=3.5, zorder=3,
                label=ARM_LABEL[_arm] if show_legend else None)
        if label_right:
            ax.annotate(ARM_LABEL[_arm], (_g["k"].iloc[-1], _g["mean"].iloc[-1]),
                        textcoords="offset points", xytext=(7, 0), fontsize=7,
                        color=_c, va="center")
    ax.axhline(0.5, color=GRID, lw=0.8, zorder=1)
    ax.set_xscale("log")
    ax.set_xlabel("labelled Indian scans (k)")
    ax.set_title(title, loc="left")
    ax.grid(axis="y"); ax.set_axisbelow(True)


for _tg, _name, _fignum in (("abnormal_any", "any reported finding (859 positive)", "F4"),
                            ("hemorrhage", "intracranial hemorrhage (94 positive)", "F4b")):
    _d = E6[E6["target"] == _tg]
    if not len(_d):
        continue
    _bbs = sorted(_d["bb"].unique())
    fig, axes = plt.subplots(1, len(_bbs), figsize=(2.5 * len(_bbs) + 1.0, 2.9), sharey=True)
    axes = np.atleast_1d(axes)
    for _i, _bb in enumerate(_bbs):
        _curve_panel(axes[_i], _d[_d["bb"] == _bb], f"{'abc'[_i]}  {_bb}",
                     show_legend=(_i == 0), label_right=(_i == len(_bbs) - 1))
    axes[0].set_ylabel("AUROC (fixed held-out 299 scans)")
    axes[0].legend(loc="upper left", handletextpad=0.4)
    fig.suptitle(f"Training from scratch on local labels - {_name}", x=0.02, y=1.04,
                 ha="left", fontsize=9.5, color=INK)
    fig.subplots_adjust(right=0.86)
    save(fig, f"{_fignum}_india_label_efficiency_{_tg}")
    plt.show()

    print(f"\nFIGURE {_fignum} NUMBERS ({_tg}) -- quantum minus parameter-matched classical")
    for _bb in _bbs:
        _s = _d[_d["bb"] == _bb]
        _p = _s.pivot_table(index="k", columns="arm", values="auroc", aggfunc="mean")
        if {"quantum", "classical"} <= set(_p.columns):
            _row = "  ".join(f"k={int(_k)}: {_p.loc[_k, 'quantum'] - _p.loc[_k, 'classical']:+.3f}"
                             for _k in _p.index)
            print(f"  {_bb:14s} {_row}")


In [ ]:
# ==============================================================================================
# FIG-6. Figure 5 -- transfer from RSNA, against training locally from scratch
# ==============================================================================================
# The comparison that matters for deployment is not "does adaptation improve on zero-shot" but
# "is the pretrained model worth anything at all next to the same labels used from scratch".
# So the transfer curve is drawn over the from-scratch curve for the same arm, same budgets,
# same fixed evaluation set, with the zero-shot value marked on the axis.
_bbs = sorted(E7["bb"].unique())
if len(_bbs):
    fig, axes = plt.subplots(1, len(_bbs), figsize=(2.5 * len(_bbs) + 1.0, 3.0), sharey=True)
    axes = np.atleast_1d(axes)
    for _i, _bb in enumerate(_bbs):
        ax = axes[_i]
        _t = E7[(E7["bb"] == _bb) & (E7["k"] > 0)]
        _f = E6[(E6["bb"] == _bb) & (E6["target"] == "hemorrhage")]
        for _arm in ("classical", "quantum"):
            _c = ARM_COLOR[_arm]
            _g = _t[_t["arm"] == _arm].groupby("k")["auroc"].mean()
            if len(_g):
                ax.plot(_g.index, _g.values, color=_c, marker="o", ms=3.5, zorder=3,
                        label=f"{ARM_LABEL[_arm]}, from RSNA" if _i == 0 else None)
            _h = _f[_f["arm"] == _arm].groupby("k")["auroc"].mean()
            if len(_h):
                ax.plot(_h.index, _h.values, color=_c, marker="s", ms=3.0, ls=(0, (4, 2)),
                        lw=1.5, alpha=0.85, zorder=3,
                        label=f"{ARM_LABEL[_arm]}, from scratch" if _i == 0 else None)
        _lp = _f[_f["arm"] == "linear_probe"].groupby("k")["auroc"].mean()
        if len(_lp):
            ax.plot(_lp.index, _lp.values, color=C_LINEAR, marker="^", ms=3.5, zorder=3,
                    label="linear probe, from scratch" if _i == 0 else None)
        _z = E7[(E7["bb"] == _bb) & (E7["k"] == 0)]
        for _, _r in _z.iterrows():
            if _r["arm"] in ARM_COLOR:
                ax.axhline(_r["auroc"], color=ARM_COLOR[_r["arm"]], lw=0.9,
                           ls=(0, (1, 2)), alpha=0.8, zorder=1)
        ax.axhline(0.5, color=GRID, lw=0.8, zorder=1)
        ax.set_xscale("log")
        ax.set_xlabel("labelled Indian scans (k)")
        ax.set_title(f"{'abc'[_i]}  {_bb}", loc="left")
        ax.grid(axis="y"); ax.set_axisbelow(True)
    axes[0].set_ylabel("AUROC, hemorrhage (fixed 299 scans)")
    axes[0].legend(loc="lower right", handletextpad=0.4, fontsize=6.5, labelspacing=0.35)
    axes[0].text(0.03, 0.97, "dotted horizontal = zero-shot", transform=axes[0].transAxes,
                 ha="left", va="top", fontsize=6.8, color=INK2)
    fig.suptitle("Transferring from RSNA vs training on the same local labels from scratch",
                 x=0.02, y=1.04, ha="left", fontsize=9.5, color=INK)
    save(fig, "F5_transfer_vs_scratch")
    plt.show()

    print("\nFIGURE 5 NUMBERS")
    for _bb in _bbs:
        _z = E7[(E7["bb"] == _bb) & (E7["k"] == 0)].set_index("arm")["auroc"]
        print(f"  {_bb}: zero-shot " + ", ".join(f"{a} {_z[a]:.3f}" for a in _z.index))
        for _k in (200, 900):
            _tt = E7[(E7["bb"] == _bb) & (E7["k"] == _k)].groupby("arm")["auroc"].mean()
            _ff = E6[(E6["bb"] == _bb) & (E6["target"] == "hemorrhage")
                     & (E6["k"] == _k)].groupby("arm")["auroc"].mean()
            for _a in ("classical", "quantum", "linear_probe"):
                if _a in _tt.index and _a in _ff.index:
                    print(f"    k={_k:<4} {_a:13s} transfer {_tt[_a]:.3f}   "
                          f"scratch {_ff[_a]:.3f}   scratch-minus-transfer "
                          f"{_ff[_a] - _tt[_a]:+.3f}")


In [ ]:
# ==============================================================================================
# FIG-7. The manuscript tables, written as CSV so nothing is retyped into the paper
# ==============================================================================================
_t1 = (E0[E0["placement"] == "pooled"]
       .groupby(["backbone", "family", "encoder_norm", "arm"])
       .agg(auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"),
            pretanh_median=("pretanh_after", "median"), n_seeds=("auroc", "size"))
       .round(4).reset_index())
_t1.to_csv(os.path.join(FIG_DIR, "table1_conditioning.csv"), index=False)

_t2 = (E0[E0["encoder_norm"] == True]                                         # noqa: E712
       .groupby(["backbone", "placement", "family", "arm"])
       .agg(auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"), n_seeds=("auroc", "size"))
       .round(4).reset_index())
_t2.to_csv(os.path.join(FIG_DIR, "table2_placement_and_arms.csv"), index=False)

_t3 = (E6.groupby(["backbone", "target", "k", "arm"])
       .agg(auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"), n_draws=("auroc", "size"))
       .round(4).reset_index())
_t3.to_csv(os.path.join(FIG_DIR, "table3_label_efficiency.csv"), index=False)

_t4 = (E7.groupby(["backbone", "mode", "k", "arm"])
       .agg(auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"), n_draws=("auroc", "size"))
       .round(4).reset_index())
_t4.to_csv(os.path.join(FIG_DIR, "table4_transfer.csv"), index=False)

print("tables written to", FIG_DIR)
for _n, _d in (("table1_conditioning", _t1), ("table2_placement_and_arms", _t2),
               ("table3_label_efficiency", _t3), ("table4_transfer", _t4)):
    print(f"  {_n}.csv  {len(_d):>5,} rows")

print("\nThe manuscript sentences are printed by the NEXT cell, after the multiplicity")
print("correction -- writing them here would quote uncorrected comparisons.")


In [ ]:
# ==============================================================================================
# FIG-8. FORMAL STATISTICS -- paired tests with multiplicity control
# ==============================================================================================
# WHY THIS CELL EXISTS. The figures above report intervals per cell, and two of them exclude
# zero: one favouring quantum, one favouring classical, on different backbones. Reported as
# "10 of 12 intervals contain zero" that is an assertion; reported without a multiplicity
# correction it is an invitation. Twelve comparisons at alpha=0.05 produce about 0.6 nominal
# hits by chance, and two hits in OPPOSITE DIRECTIONS is the signature of noise rather than a
# real effect. So every claim in the paper that is a comparison is tested here, in named
# families, with Benjamini-Hochberg applied WITHIN each family, and the direction of each
# nominal hit is printed so a reader can see the inconsistency for themselves.
#
# The test is paired and the pairing is the point: within a cell the two arms see identical
# splits, identical seeds (E0) or identical label draws (E6/E7), so test-set sampling is
# common to both sides and cancels in the difference. What is left is the variation the claim
# is actually about.
from scipy import stats as _st


def _bh(p):
    """Benjamini-Hochberg q-values."""
    p = np.asarray(p, dtype=float)
    n = len(p)
    o = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0
    for r in range(n - 1, -1, -1):
        prev = min(prev, p[o[r]] * n / (r + 1))
        q[o[r]] = prev
    return q


def paired_family(df, pair_on, group_cols, arm_a, arm_b, label_fmt):
    """One comparison family: paired differences within `pair_on`, t-test, then BH."""
    rows = []
    for _k, _s in df.groupby(group_cols):
        _a = _s[_s["arm"] == arm_a].groupby(pair_on)["auroc"].mean()
        _b = _s[_s["arm"] == arm_b].groupby(pair_on)["auroc"].mean()
        _d = (_a - _b).dropna()
        if len(_d) < 3:
            continue
        _t, _p = _st.ttest_1samp(_d, 0.0)
        _se = _d.std(ddof=1) / np.sqrt(len(_d))
        _c = _st.t.ppf(0.975, len(_d) - 1) * _se
        rows.append(dict(cell=label_fmt(_k), delta=_d.mean(), lo=_d.mean() - _c,
                         hi=_d.mean() + _c, p=float(_p), n=len(_d)))
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    out["q"] = _bh(out["p"])
    out["sig_raw"] = out["p"] < 0.05
    out["sig_bh"] = out["q"] < 0.05
    return out.sort_values("p").reset_index(drop=True)


def report(name, tab, claim):
    print("\n" + "=" * 94)
    print(name)
    print("=" * 94)
    if not len(tab):
        print("  no cells with enough pairs.")
        return
    print(tab[["cell", "delta", "lo", "hi", "p", "q", "n"]].round(4).to_string(index=False))
    _nr, _nb = int(tab["sig_raw"].sum()), int(tab["sig_bh"].sum())
    _exp = 0.05 * len(tab)
    print(f"\n  {_nr}/{len(tab)} nominally significant (expected ~{_exp:.1f} by chance); "
          f"{_nb}/{len(tab)} survive Benjamini-Hochberg.")
    if _nr:
        _pos = int((tab["sig_raw"] & (tab["delta"] > 0)).sum())
        _neg = int((tab["sig_raw"] & (tab["delta"] < 0)).sum())
        print(f"  directions among the nominal hits: {_pos} positive, {_neg} negative"
              + ("  <- OPPOSITE SIGNS, which is what noise looks like"
                 if _pos and _neg else ""))
    print(f"  largest |difference| anywhere: {tab['delta'].abs().max():.4f}")
    print(f"  -> {claim}")


_cond = E0[E0["encoder_norm"] == True]                                        # noqa: E712

A = paired_family(_cond, "seed", ["bb", "placement", "family"], "quantum", "classical",
                  lambda k: f"{k[0].split()[0]} {'spatial' if k[1] == 'spatial' else k[2].split('_')[0]}")
report("FAMILY A -- quantum minus parameter-matched classical (RSNA, conditioned)", A,
       "the variational circuit is not distinguishable from a classical tail of the same size.")

_pl = []
for _bb in sorted(_cond["bb"].unique()):
    _sp = _cond[(_cond["bb"] == _bb) & (_cond["placement"] == "spatial")
                & (_cond["arm"] == "quantum")].groupby("seed")["auroc"].mean()
    _po = _cond[(_cond["bb"] == _bb) & (_cond["placement"] == "pooled")
                & (_cond["family"] == "Q1_sel") & (_cond["arm"] == "quantum")
                ].groupby("seed")["auroc"].mean()
    _d = (_sp - _po).dropna()
    if len(_d) >= 3:
        _t, _p = _st.ttest_1samp(_d, 0.0)
        _se = _d.std(ddof=1) / np.sqrt(len(_d))
        _c = _st.t.ppf(0.975, len(_d) - 1) * _se
        _pl.append(dict(cell=_bb, delta=_d.mean(), lo=_d.mean() - _c, hi=_d.mean() + _c,
                        p=float(_p), n=len(_d)))
B = pd.DataFrame(_pl)
if len(B):
    B["q"] = _bh(B["p"]); B["sig_raw"] = B["p"] < 0.05; B["sig_bh"] = B["q"] < 0.05
report("FAMILY B -- 8 spatial sites minus 1 pooled site (both conditioned)", B,
       "placement effects are small and INCONSISTENT IN SIGN across backbones. Where one "
       "survives multiplicity it favours the SIMPLER placement, not the deeper one. So the "
       "defensible sentence is: 'applying the circuit at eight spatial sites rather than one "
       "did not improve discrimination on any backbone, and significantly reduced it on one'. "
       "Do NOT write 'placement has no effect' -- one cell survives BH and a reader who "
       "checks the table will see it.")

C = paired_family(E6, "draw", ["bb", "target", "k"], "quantum", "classical",
                  lambda k: f"{k[0].split()[0]} {k[1][:9]} k={int(k[2])}")
report("FAMILY C -- quantum minus classical on the Indian cohort, by label budget", C,
       "this is the published low-data claim, tested directly. Read the SIGN column: a "
       "claim of greater robustness predicts positive deltas that GROW as k shrinks.")
if len(C):
    _small = C[C["cell"].str.contains("k=25|k=50")]
    _large = C[C["cell"].str.contains("k=900")]
    print(f"\n  mean delta at k=25/50 : {_small['delta'].mean():+.4f}  "
          f"({int((_small['delta'] < 0).sum())}/{len(_small)} favour classical)")
    print(f"  mean delta at k=900   : {_large['delta'].mean():+.4f}  "
          f"({int((_large['delta'] < 0).sum())}/{len(_large)} favour classical)")
    print("  The deficit is LARGEST where labels are scarcest, which is the opposite of the")
    print("  claim under test.")

_stats = {}
for _n, _t in (("A_quantum_vs_classical", A), ("B_placement", B), ("C_india_label_budget", C)):
    if len(_t):
        _t.to_csv(os.path.join(FIG_DIR, f"stats_{_n}.csv"), index=False)
        _stats[_n] = _t
print(f"\nstatistics written to {FIG_DIR}: " + ", ".join(f"stats_{k}.csv" for k in _stats))


# ==============================================================================================
# SENTENCES YOU CAN WRITE -- generated AFTER the correction, never before
# ==============================================================================================
# An earlier version of this notebook printed these before the multiplicity step, and one of
# them read "10 of 12 intervals contain zero" -- which is the uncorrected claim and exactly
# what a reviewer will challenge. They are generated here instead, from the corrected tables.
_pl = E0[E0["placement"] == "pooled"]
_wq = _pl[_pl["arm"] == "quantum"].pivot_table(
    index=["backbone", "family"], columns="encoder_norm", values="auroc")
_liftq = (_wq[True] - _wq[False]).dropna()
_pt0 = _pl[_pl["encoder_norm"] == False]["pretanh_after"]
_pt1 = _pl[_pl["encoder_norm"] == True]["pretanh_after"]

print("\n" + "=" * 94)
print("SENTENCES YOU CAN WRITE, WITH THE CORRECTED NUMBERS IN THEM")
print("=" * 94)
print(f"""
1. MECHANISM. Without normalisation the learned angle encoder drives the pre-activation to a
   median |z| of {_pt0.median():.0f} (range {_pt0.min():.0f}-{_pt0.max():.0f}), where tanh is flat to machine precision. A
   single non-affine BatchNorm with no trainable parameters returns it to {_pt1.median():.2f}
   ({_pt1.min():.2f}-{_pt1.max():.2f}) and raises QUANTUM-arm AUROC by {_liftq.mean():+.3f} on average
   (range {_liftq.min():+.3f} to {_liftq.max():+.3f}) across {len(_liftq)} backbone x ansatz cells and {int(E0['seed'].nunique())} seeds.
   [say "quantum-arm" every time -- the all-arms mean is different and the two get confused]

2. ANSATZ. Unconditioned, the spread across ansatze is {_R.loc[_R['norm'] == False, 'spread'].mean():.3f}, or
   {(_R.loc[_R['norm'] == False, 'spread'] / _R.loc[_R['norm'] == False, 'seed_sd']).mean():.1f}x the seed-to-seed standard deviation; conditioned it is
   {_R.loc[_R['norm'] == True, 'spread'].mean():.3f}, or {(_R.loc[_R['norm'] == True, 'spread'] / _R.loc[_R['norm'] == True, 'seed_sd']).mean():.1f}x. The apparent ranking of ansatze was a ranking of how
   quickly each drives its encoder into saturation.

3. PLACEMENT. Applying the circuit at eight spatial sites rather than one did not improve
   discrimination on any backbone, and significantly reduced it on one
   ({B.loc[B['delta'].idxmin(), 'cell']}, {B['delta'].min():+.3f} [{B.loc[B['delta'].idxmin(), 'lo']:+.3f}, {B.loc[B['delta'].idxmin(), 'hi']:+.3f}], q={B.loc[B['delta'].idxmin(), 'q']:.3f}).
   [do NOT write "no effect" -- {int(B['sig_bh'].sum())} of {len(B)} survives correction]

4. QUANTUM vs CLASSICAL. Across {len(A)} parameter-matched comparisons the difference was
   {A['delta'].mean():+.4f} on average (largest |difference| {A['delta'].abs().max():.4f}); {int(A['sig_raw'].sum())} were nominally
   significant against {0.05 * len(A):.1f} expected by chance, they pointed in opposite directions, and
   {int(A['sig_bh'].sum())} survived Benjamini-Hochberg.

5. LABEL BUDGET (the refutation). Across {len(C)} paired comparisons on the Indian cohort the
   quantum arm was below its parameter-matched classical control by {C['delta'].mean():+.4f} on
   average; {int(C['sig_raw'].sum())} were nominally significant and {int(C['sig_bh'].sum())} survived Benjamini-Hochberg, with
   {int((C['sig_raw'] & (C['delta'] > 0)).sum())} of the nominal hits favouring quantum. The deficit was larger at the smallest
   budgets ({_small['delta'].mean():+.4f} at k=25-50) than at the largest ({_large['delta'].mean():+.4f} at k=900) -- the
   opposite of the direction the low-data-robustness claim predicts.
""")


In [ ]:
# ==============================================================================================
# FIG-9. EQUIVALENCE TESTING AND A SIGN TEST -- the two analyses a null claim actually needs
# ==============================================================================================
# A failure to reject is not evidence of equivalence. Two additions close that gap.
#
# 1. TWO ONE-SIDED TESTS (TOST) for Family A, against a margin fixed in the Methods rather than
#    chosen after seeing the data. If both one-sided tests reject at the margin, the arms are
#    statistically EQUIVALENT within it -- a positive claim, not an absence of one.
#
# 2. A CELL-LEVEL BINOMIAL SIGN TEST for Family C. Within a label budget the 50 draws are not
#    independent at large k (each draw is nearly the whole pool), so those p-values are
#    optimistic. The sign of each cell's point estimate does not depend on that structure, and
#    36 cells spanning three backbones, two endpoints and six budgets are a far weaker
#    assumption. This is the analysis the label-budget conclusion should rest on.
from scipy import stats as _st

EQUIV_MARGIN = 0.02      # stated in Methods; not chosen after seeing the result

print("=" * 94)
print(f"EQUIVALENCE (TOST) -- Family A, margin +/-{EQUIV_MARGIN} AUROC")
print("=" * 94)
_rows = []
for _k, _s in _cond.groupby(["bb", "placement", "family"]):
    _q = _s[_s["arm"] == "quantum"].groupby("seed")["auroc"].mean()
    _c = _s[_s["arm"] == "classical"].groupby("seed")["auroc"].mean()
    _d = (_q - _c).dropna()
    if len(_d) < 3:
        continue
    _n = len(_d)
    _se = _d.std(ddof=1) / np.sqrt(_n)
    if _se == 0:
        continue
    # H0 for each one-sided test: the difference is AT LEAST as extreme as the margin
    _p_lo = _st.t.sf((_d.mean() + EQUIV_MARGIN) / _se, _n - 1)      # tests delta > -margin
    _p_hi = _st.t.cdf((_d.mean() - EQUIV_MARGIN) / _se, _n - 1)     # tests delta < +margin
    _p = max(_p_lo, _p_hi)
    _rows.append(dict(
        cell=f"{_k[0].split()[0]} {'spatial' if _k[1] == 'spatial' else _k[2].split('_')[0]}",
        delta=_d.mean(), lo90=_d.mean() - _st.t.ppf(0.95, _n - 1) * _se,
        hi90=_d.mean() + _st.t.ppf(0.95, _n - 1) * _se, p_tost=_p,
        equivalent=bool(_p < 0.05), n=_n))
EQ = pd.DataFrame(_rows).sort_values("p_tost").reset_index(drop=True)
if len(EQ):
    print(EQ.round(4).to_string(index=False))
    _ne = int(EQ["equivalent"].sum())
    print(f"\n  {_ne}/{len(EQ)} comparison(s) are statistically EQUIVALENT within "
          f"+/-{EQUIV_MARGIN}.")
    print("  The 90% interval is the one to read against the margin: TOST at alpha=0.05 is")
    print("  equivalent to asking whether the 90% interval lies entirely inside it.")
    if _ne == len(EQ):
        print("  -> You may write: 'quantum and parameter-matched classical adapters were")
        print("     statistically equivalent within a pre-specified margin of 0.02 AUROC.'")
        print("     That is a positive claim and is much stronger than a failure to reject.")
    else:
        _no = EQ[~EQ["equivalent"]]["cell"].tolist()
        print(f"  -> NOT equivalent in: {_no}. Name them. Write 'equivalent in {_ne} of")
        print(f"     {len(EQ)} comparisons; in the remainder the data are consistent with no")
        print("     difference but do not establish equivalence at this margin.'")
    EQ.to_csv(os.path.join(FIG_DIR, "stats_A_equivalence.csv"), index=False)

print("\n" + "=" * 94)
print("SIGN TEST -- Family C, cell-level point estimates")
print("=" * 94)
if len(C):
    _neg = int((C["delta"] < 0).sum())
    _pos = int((C["delta"] > 0).sum())
    _n = _neg + _pos
    _p = _st.binomtest(_neg, _n, 0.5, alternative="two-sided").pvalue
    print(f"  {_neg}/{_n} cell-level point estimates favour the classical arm "
          f"({_pos} favour quantum)")
    print(f"  two-sided binomial p = {_p:.3g}")
    print("  This makes no assumption about dependence between draws within a budget, which is")
    print("  the weakness of the paired p-values at large k. It is the statement the")
    print("  label-budget conclusion should rest on.")
    pd.DataFrame([dict(n_cells=_n, n_favour_classical=_neg, n_favour_quantum=_pos,
                       binomial_p_two_sided=_p)]).to_csv(
        os.path.join(FIG_DIR, "stats_C_sign_test.csv"), index=False)
    _small = C[C["cell"].str.contains("k=25|k=50")]
    if len(_small):
        _ns = int((_small["delta"] < 0).sum())
        _ps = _st.binomtest(_ns, len(_small), 0.5, alternative="two-sided").pvalue
        print(f"\n  restricted to the smallest budgets (k=25, k=50): {_ns}/{len(_small)} "
              f"favour classical, p = {_ps:.3g}")
        print("  The published claim predicts the opposite at exactly these budgets.")
print("\nInsert both results into Sections 4.4 and 4.7 where the manuscript marks them.")
